# Residential Automated Valuation Model (AVM) — End-to-End Linear Regression Engine
### A Senior ML Engineering & Econometric Modeling Portfolio Project
---
This notebook demonstrates the complete machine learning workflow centered on **Linear Regression and Regularized Variants (Ridge, Lasso, ElasticNet)** for property appraisal.

**Sections:**
1. Baseline Zero-Information Models
2. Exploratory Data Analysis & Anomaly Detection
3. Leakage-Free Scikit-Learn Pipelines
4. 5-Fold Cross-Validation Grid Search & Non-Linear Benchmark
5. Gauss-Markov Assumption Diagnostics & VIF
6. Non-Technical Executive Interpretations


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import statsmodels.api as sm

from src.data.loader import load_raw_data
from src.models.baseline import evaluate_baselines
from src.features.transformers import build_full_feature_pipeline
from src.utils.metrics import evaluate_regression
from src.utils.diagnostics import compute_vif, run_assumption_diagnostics
from src.models.validation_and_interpretation import extract_business_interpretations


## 1. Load Data & Baseline Model Benchmark
Zero-information naive predictors establish the lower bound of acceptable predictive utility.

In [ ]:
df = load_raw_data()
print(f"Raw dataset shape: {df.shape}")
mean_res, median_res = evaluate_baselines(df)
print("Naive Mean Regressor Test Results:", mean_res)
print("Naive Median Regressor Test Results:", median_res)


## 2. Load Production Artifact & Evaluate on Holdout Split

In [ ]:
model_pipeline = joblib.load("models/best_linear_pipeline.joblib")
test_df = pd.read_parquet("data/processed/test.parquet")
X_test = test_df.drop(columns=["SalePrice"])
y_test = test_df["SalePrice"].values
y_pred = model_pipeline.predict(X_test)

metrics = evaluate_regression(y_test, y_pred, p=X_test.shape[1])
print("Production Ridge Model Performance on Holdout Set:")
for k, v in metrics.items():
    print(f"  - {k}: {v}")


## 3. Top Drivers of Residential Value (Linear Coefficient Interpretation)

In [ ]:
interps = extract_business_interpretations(model_pipeline)
interp_df = pd.DataFrame(interps)
interp_df[["feature", "standardized_coefficient", "approx_price_impact_pct", "business_takeaway"]].head(8)
